# Attestor -- Owen Coder Training (3B)
# *for AI's, by AI*

Fine-tune **Qwen/Qwen2.5-Coder-3B-Instruct** on Attestor's security training data.
Produces a GGUF model ready for Ollama deployment.

**Config:** LoRA r=64 alpha=128, 6 epochs, lr=0.0001, seq_len=4096


In [ ]:
# Step 1: Install dependencies
!pip install -q 'unsloth[colab-new]' datasets trl


In [ ]:
# Step 2: Upload training data
# Upload training_data_merged.jsonl (and optionally feedback_training_data.jsonl)
from google.colab import files
uploaded = files.upload()


In [ ]:
# Step 3: Load and merge training data
import json, os

rows = []
for fn in sorted(uploaded.keys()):
    if fn.endswith('.jsonl'):
        with open(fn) as f:
            batch = [json.loads(l) for l in f if l.strip()]
        rows.extend(batch)
        print(f'  {fn}: {len(batch)} pairs')

# Deduplicate
import hashlib
seen = set()
unique = []
for r in rows:
    key = hashlib.md5(r['instruction'].strip().encode()).hexdigest()
    if key not in seen:
        seen.add(key)
        unique.append(r)
rows = unique
print(f'\nTotal: {len(rows)} unique training pairs')


In [ ]:
# Step 4: Load model with QLoRA
from unsloth import FastLanguageModel

BASE_MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct'
MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128, lora_dropout=0.05, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')


In [ ]:
# Step 5: Format dataset
from datasets import Dataset

SYSTEM = 'You are a security analysis engine. You find vulnerabilities in source code with high precision. You report findings with specific CWE IDs, severity ratings, line numbers, and concrete fix suggestions. You never hallucinate vulnerabilities that don\'t exist.'

TEMPLATE = '<|im_start|>system\n{system}<|im_end|>\n<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>'

dataset = Dataset.from_list([{
    'text': TEMPLATE.format(
        system=SYSTEM,
        instruction=r['instruction'],
        output=r['output']
    )
} for r in rows])

print(f'Dataset: {len(dataset)} examples')
print(f'Sample length: {len(dataset[0]["text"])} chars')


In [ ]:
# Step 6: Train
import time
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=MAX_SEQ,
    dataset_num_proc=2, packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.06, num_train_epochs=6,
        learning_rate=0.0001, lr_scheduler_type='cosine',
        weight_decay=0.01, fp16=True, bf16=False,
        logging_steps=5, optim='adamw_8bit', seed=42,
        output_dir='owen-coder-lora',
        save_strategy='epoch', report_to='none'))

t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0
print(f'Loss: {stats.training_loss:.4f}')
print(f'Time: {elapsed:.0f}s ({elapsed/60:.1f}m)')


In [ ]:
# Step 7: Export GGUF
model.save_pretrained_gguf(
    'owen-coder-merged', tokenizer,
    quantization_method='q4_k_m')

# Create Modelfile for Ollama
import glob
gguf = glob.glob('owen-coder-merged/*.gguf')[0]
gguf_name = os.path.basename(gguf)

with open('owen-coder-merged/Modelfile', 'w') as f:
    f.write(f'FROM ./{gguf_name}\n\n')
    f.write(f'SYSTEM \"\"\"You are a security analysis engine. You find vulnerabilities in source code with high precision. You report findings with specific CWE IDs, severity ratings, line numbers, and concrete fix suggestions. You never hallucinate vulnerabilities that don\'t exist.\"\"\"\n\n')
    f.write('PARAMETER temperature 0.1\n')
    f.write('PARAMETER top_p 0.9\n')
    f.write('PARAMETER num_ctx 4096\n')
    f.write('PARAMETER stop "<|im_end|>"\n')

print(f'GGUF exported: {gguf}')
print('Modelfile created')


In [ ]:
# Step 8: Download
import shutil
shutil.make_archive('owen-coder-merged', 'zip', 'owen-coder-merged')
files.download('owen-coder-merged.zip')

print('\n--- DEPLOYMENT ---')
print('1. Unzip owen-coder-merged.zip')
print('2. cd owen-coder-merged')
print('3. ollama create owen-coder -f Modelfile')
print('4. ollama run owen-coder')


# Owen Coder is ready!
# Deploy with: `ollama create owen-coder -f Modelfile`
# Then run Attestor with the trained model for hybrid analysis.
